# Chowdeck — Data Cleaning + EDA

Notebook làm sạch dữ liệu và phân tích khám phá (EDA) cho dataset Chowdeck (Nigeria) — dataset chính của dự án.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
pd.set_option('display.max_columns', 50)

## 0. ĐƯỜNG DẪN (tự động tìm đúng thư mục gốc project, chạy từ đâu cũng được)

In [4]:
try:
    BASE_DIR = Path(__file__).resolve().parent.parent  # khi chạy dạng .py
except NameError:
    BASE_DIR = Path.cwd().resolve().parent  # khi chạy trong Jupyter Notebook (đứng trong notebooks/)
RAW_PATH = BASE_DIR / 'data' / 'raw' / 'Chowdeck_Order_Delivery_Details.xlsx'
CLEAN_DIR = BASE_DIR / 'data' / 'clean'
CHARTS_DIR = BASE_DIR / 'outputs' / 'charts'
KPI_DIR = BASE_DIR / 'outputs' / 'kpi_summary'
for d in [CLEAN_DIR, CHARTS_DIR, KPI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. LOAD DATA

In [5]:
df = pd.read_excel(RAW_PATH, sheet_name='Sheet1')
print("Shape ban đầu:", df.shape)

Shape ban đầu: (5000, 27)


## 2. CHUẨN HÓA TÊN CỘT

In [6]:
df.columns = [c.strip() for c in df.columns]  # bỏ khoảng trắng thừa (vd 'Total ' -> 'Total')

## 3. KIỂM TRA TÍNH LOGIC THỜI GIAN

In [7]:
time_cols = ['Order Received', 'Preparing Order', 'Rider Accepted', 'Order Ready',
             'Rider At Vendor', 'Rider Picked Up', 'Order Arrived', 'Order Delivered']

# Đánh dấu dòng có timestamp không tăng dần đúng thứ tự
is_monotonic = pd.Series(True, index=df.index)
for i in range(len(time_cols) - 1):
    is_monotonic &= (df[time_cols[i]] <= df[time_cols[i + 1]])
df['time_sequence_valid'] = is_monotonic
print(f"\nSố đơn có thứ tự thời gian KHÔNG hợp lệ: {(~is_monotonic).sum()} / {len(df)}")


Số đơn có thứ tự thời gian KHÔNG hợp lệ: 1463 / 5000


## 4. KIỂM TRA OUTLIER (chỉ xem, chưa loại bỏ)

In [8]:
print("\nThống kê mô tả các cột số:")
print(df[['Unit Price', 'Distance (km)', 'Sub Total', 'Total']].describe())


Thống kê mô tả các cột số:
         Unit Price  Distance (km)      Sub Total          Total
count   5000.000000    5000.000000    5000.000000    5000.000000
mean    6762.200000       8.518440    8886.800000   10280.927600
std     7334.624087       4.143014   10605.790037   10776.467286
min      500.000000       0.500000     500.000000    1160.000000
25%     2000.000000       5.000000    3000.000000    3856.000000
50%     5000.000000       8.200000    5000.000000    6428.500000
75%     7000.000000      12.100000   10000.000000   12012.750000
max    35000.000000      16.000000  105000.000000  106675.000000


## 5. KIỂM TRA RATING HỢP LỆ

In [9]:
invalid_rating = df[(df['Rating'] < 1) | (df['Rating'] > 5)]
print(f"\nSố Rating ngoài khoảng [1,5]: {len(invalid_rating)}")


Số Rating ngoài khoảng [1,5]: 0


## 6. TẠO CỘT PHÁI SINH: THỜI GIAN XỬ LÝ

In [10]:
df['prep_time_min'] = (df['Order Ready'] - df['Preparing Order']).dt.total_seconds() / 60
df['delivery_time_min'] = (df['Order Delivered'] - df['Order Received']).dt.total_seconds() / 60
df['delay_min'] = (df['Order Delivered'] - df['Expected Delivery Time']).dt.total_seconds() / 60
# delay_min > 0 : giao trễ so với dự kiến | < 0 : giao sớm hơn dự kiến

## 7. TẠO CỘT PHÁI SINH: THỜI ĐIỂM ĐẶT HÀNG

In [11]:
df['order_hour'] = df['Order Received'].dt.hour
df['order_dayofweek'] = df['Order Received'].dt.day_name()
df['order_month'] = df['Order Received'].dt.to_period('M').astype(str)
df['order_year'] = df['Order Received'].dt.year

## 8. XÁC ĐỊNH CỘT REVENUE CHUẨN

In [12]:
df['revenue'] = df['Sub Total']  # Revenue của nhà hàng, KHÔNG gồm Delivery Fee/Service Fee

## 8b. LỌC PHẠM VI: CHỈ GIỮ NHÓM LIÊN QUAN ẨM THỰC (theo xác nhận của user)

In [13]:
restaurant_categories = ['Food', 'Drinks & Beverages', 'Pastries']
n_before = len(df)
df = df[df['Order Category'].isin(restaurant_categories)].copy()
n_after = len(df)
print(f"\nLọc Order Category: giữ {n_after}/{n_before} dòng "
      f"(loại bỏ Groceries & Medications, mất {n_before - n_after} dòng)")


Lọc Order Category: giữ 3061/5000 dòng (loại bỏ Groceries & Medications, mất 1939 dòng)


## 9. LOẠI CỘT KHÔNG PHỤC VỤ PHÂN TÍCH

In [14]:
df_clean = df.drop(columns=['Delivery PIN', 'Url', 'Wallet Balance'])

## 10. LƯU FILE ĐÃ LÀM SẠCH

In [15]:
df_clean.to_csv(CLEAN_DIR / 'chowdeck_clean.csv', index=False)
print(f"\nĐã lưu chowdeck_clean.csv | Shape cuối: {df_clean.shape}")

# ============================================================
# EDA - PHÂN TÍCH KHÁM PHÁ DỮ LIỆU
# ============================================================

# --- Q1: Nhà hàng nào tạo doanh thu & AOV cao nhất? ---
shop_stats = df_clean.groupby('Shop Name').agg(
    total_orders=('revenue', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_order_value=('Total', 'mean'),
    avg_rating=('Rating', 'mean')
).sort_values('total_revenue', ascending=False)
print("\n=== Top 10 nhà hàng theo doanh thu ===")
print(shop_stats.head(10))

plt.figure(figsize=(10, 6))
shop_stats['total_revenue'].head(10).plot(kind='barh')
plt.title('Top 10 Nhà hàng theo Tổng Doanh thu (Sub Total)')
plt.xlabel('Doanh thu (₦)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_top10_shops_revenue.png', dpi=120)
plt.close()

# --- Q2: Nhóm món ăn nào đóng góp doanh thu nhiều nhất? ---
category_revenue = df_clean.groupby('Order Category')['revenue'].sum().sort_values(ascending=False)
print("\n=== Doanh thu theo Order Category ===")
print(category_revenue)

plt.figure(figsize=(8, 5))
category_revenue.plot(kind='bar', color='teal')
plt.title('Doanh thu theo Nhóm sản phẩm (Order Category)')
plt.ylabel('Doanh thu (₦)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_revenue_by_category.png', dpi=120)
plt.close()

# --- Q3: Rating có liên hệ với số đơn / AOV không? (ở cấp nhà hàng) ---
corr_rating_orders = shop_stats['avg_rating'].corr(shop_stats['total_orders'])
corr_rating_aov = shop_stats['avg_rating'].corr(shop_stats['avg_order_value'])
print(f"\nTương quan Rating trung bình vs Số đơn hàng (theo nhà hàng): {corr_rating_orders:.3f}")
print(f"Tương quan Rating trung bình vs AOV (theo nhà hàng): {corr_rating_aov:.3f}")

plt.figure(figsize=(7, 5))
sns.scatterplot(data=shop_stats, x='avg_rating', y='total_orders')
plt.title('Rating trung bình vs Số lượng đơn hàng (theo Nhà hàng)')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_rating_vs_orders.png', dpi=120)
plt.close()

# --- Q4: Thời gian giao hàng/chuẩn bị món có liên hệ với Rating/số đơn? ---
corr_delivery_rating = df_clean['delivery_time_min'].corr(df_clean['Rating'])
corr_prep_rating = df_clean['prep_time_min'].corr(df_clean['Rating'])
print(f"\nTương quan Thời gian giao hàng vs Rating (cấp đơn hàng): {corr_delivery_rating:.3f}")
print(f"Tương quan Thời gian chuẩn bị món vs Rating (cấp đơn hàng): {corr_prep_rating:.3f}")

# --- Q5: Khu vực nào có doanh thu & mật độ đơn cao nhất? ---
location_stats = df_clean.groupby('Delivery Location').agg(
    total_orders=('revenue', 'count'),
    total_revenue=('revenue', 'sum')
).sort_values('total_revenue', ascending=False)
print("\n=== Doanh thu theo Khu vực giao hàng ===")
print(location_stats)

# --- Q6: Khung giờ/ngày nào tạo doanh thu cao nhất? ---
hourly_revenue = df_clean.groupby('order_hour')['revenue'].sum()
dow_revenue = df_clean.groupby('order_dayofweek')['revenue'].sum().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

plt.figure(figsize=(10, 5))
hourly_revenue.plot(kind='line', marker='o')
plt.title('Doanh thu theo Khung giờ trong ngày')
plt.xlabel('Giờ')
plt.ylabel('Doanh thu (₦)')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_revenue_by_hour.png', dpi=120)
plt.close()

plt.figure(figsize=(8, 5))
dow_revenue.plot(kind='bar', color='coral')
plt.title('Doanh thu theo Thứ trong tuần')
plt.ylabel('Doanh thu (₦)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_revenue_by_dayofweek.png', dpi=120)
plt.close()

# --- Tính KPI tổng hợp để lưu vào summary_metrics ---
kpi_summary = {
    'dataset': 'Chowdeck',
    'market': 'Nigeria',
    'total_orders': len(df_clean),
    'total_revenue': df_clean['revenue'].sum(),
    'aov': df_clean['Total'].mean(),
    'avg_rating': df_clean['Rating'].mean(),
    'corr_rating_vs_orders(shop_level)': corr_rating_orders,
    'corr_rating_vs_aov(shop_level)': corr_rating_aov,
    'corr_delivery_time_vs_rating': corr_delivery_rating,
    'corr_prep_time_vs_rating': corr_prep_rating,
}
pd.DataFrame([kpi_summary]).to_csv(KPI_DIR / 'chowdeck_kpi_summary.csv', index=False)

print("\n=== HOÀN TẤT CLEANING + EDA CHOWDECK ===")
print("Các file đã tạo: chowdeck_clean.csv, chowdeck_kpi_summary.csv, và 5 biểu đồ PNG")


Đã lưu chowdeck_clean.csv | Shape cuối: (3061, 33)

=== Top 10 nhà hàng theo doanh thu ===
                    total_orders  total_revenue  avg_order_value  avg_rating
Shop Name                                                                   
Tasty Bites                  207        2293000     12457.367150    3.927536
Mama Cass Kitchen            192        2176500     12721.994792    3.833333
Naija Kitchen                188        2143000     12818.643617    3.914894
Kulture Kitchen              200        2139000     12046.105000    3.935000
Lekki Eats                   198        2123000     12140.429293    3.914141
Sweet Bites Bakery           214        1947000     10208.822430    3.855140
Oven Fresh                   211        1915000     10193.004739    3.876777
BakeHouse                    202        1785000      9955.039604    4.004950
Golden Crust                 216        1768500      9276.203704    3.916667
Sunrise Bakery               182        1405000      8811.543